# Jupyter Notebook Tutorial
## A hands-on walkthrough with pandas examples

Run each cell with **Shift + Enter**. Read the markdown cells for explanation.

---

## 1 · Import Libraries
Always start your notebook by importing everything you need.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline   # makes plots appear inside the notebook

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print("Ready!")

## 2 · DataFrames Render as Tables

Put a DataFrame as the **last line** of a cell — no `print()` needed.
Jupyter automatically renders it as a formatted HTML table.

In [ ]:
df = pd.DataFrame({
    'employee_id': ['C1234', 'F5678', 'C9012', 'F3456', 'C7890'],
    'department':  ['Quant', 'Risk',  'Quant', 'Tech',  'Risk'],
    'month':       ['2025-01', '2025-01', '2025-02', '2025-02', '2025-02'],
    'cost':        [1200.0, 850.0, 2100.0, 450.0, 3200.0],
    'cpu_hours':   [120, 80, 210, 45, 310],
    'gpu_hours':   [10,   0,  30,  5,  50],
})

df   # last line = auto-rendered as table

## 3 · Multiple Outputs in One Cell
`print()` statements appear above the final rendered table.

In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print()
df.describe().round(2)

## 4 · Tab Completion & Help

- Type `df.` and press **Tab** → see all available methods
- Place your cursor inside a function call and press **Shift+Tab** → see docs

Try it in the cell below:

In [ ]:
# Uncomment and press Tab after the dot:
# df.

# Place cursor inside the parentheses and press Shift+Tab:
# df.groupby(

## 5 · Magic Commands
Special `%` commands that control the Jupyter environment.

In [ ]:
# Time a single execution
%time df.sort_values('cost')

In [ ]:
# Time many runs and report average
%timeit df['cost'].sum()

In [ ]:
# See all variables currently in memory
%whos

## 6 · Inline Plots
Charts render directly in the notebook — no separate window needed.

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(df['employee_id'], df['cost'],
        color=['#4472C4','#ED7D31','#4472C4','#ED7D31','#4472C4'])
plt.title('Cost by Employee', fontsize=13)
plt.xlabel('Employee ID')
plt.ylabel('Cost ($)')
plt.tight_layout()
plt.show()

## 7 · Multiple Subplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: bar chart
axes[0].bar(df['employee_id'], df['cost'])
axes[0].set_title('Cost by Employee')
axes[0].set_xlabel('Employee ID')
axes[0].set_ylabel('Cost ($)')

# Right: pie chart
dept_cost = df.groupby('department')['cost'].sum()
axes[1].pie(dept_cost, labels=dept_cost.index,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Cost Share by Department')

plt.suptitle('EMR Cluster Cost Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8 · Styled DataFrame Output

Pandas `.style` lets you add colour, formatting, and captions to tables.
This only works in Jupyter — it renders as HTML.

In [ ]:
summary = df.groupby('department').agg(
    users      = ('employee_id', 'nunique'),
    total_cost = ('cost',        'sum'),
    avg_cost   = ('cost',        'mean'),
    total_cpu  = ('cpu_hours',   'sum'),
).round(2)

summary.style \
    .format({'total_cost': '${:,.0f}', 'avg_cost': '${:,.0f}'}) \
    .background_gradient(subset=['total_cost'], cmap='Blues') \
    .set_caption('Cost Summary by Department')

## 9 · Filtering and Groupby

In [ ]:
# Filter: only Quant and Risk, cost over $1000
filtered = df[
    df['department'].isin(['Quant', 'Risk']) &
    (df['cost'] > 1000)
]
filtered[['employee_id', 'department', 'cost']]

In [ ]:
# Groupby with multiple aggregations
df.groupby(['department', 'month']).agg(
    total_cost = ('cost',      'sum'),
    num_users  = ('employee_id','nunique'),
).round(2)

## 10 · Pivot Table

In [ ]:
pivot = df.pivot_table(
    index      = 'employee_id',
    columns    = 'month',
    values     = 'cost',
    aggfunc    = 'sum',
    fill_value = 0,
)

# Add month-over-month change
months = pivot.columns.tolist()
for i in range(1, len(months)):
    prev, curr = months[i-1], months[i]
    pivot[f'Δ {curr}'] = pivot[curr] - pivot[prev]

pivot

## 11 · Adding Columns

In [ ]:
df['total_hours'] = df['cpu_hours'] + df['gpu_hours']
df['cost_per_hr'] = (df['cost'] / df['total_hours']).round(2)
df['cost_pct']    = (df['cost'] / df['cost'].sum() * 100).round(1)
df['band']        = df['cost'].apply(
    lambda x: 'High' if x >= 2000 else 'Mid' if x >= 1000 else 'Low'
)

df[['employee_id', 'cost', 'total_hours', 'cost_per_hr', 'cost_pct', 'band']]

## 12 · Handling Missing Data

In [ ]:
df2 = df[['employee_id', 'cost', 'gpu_hours']].copy()
df2.loc[1, 'cost']      = None   # introduce missing values
df2.loc[3, 'gpu_hours'] = None

print("Null counts:")
print(df2.isnull().sum())
print()

# Fill missing values
df2_filled = df2.fillna({
    'cost':      df2['cost'].median(),
    'gpu_hours': 0,
})
df2_filled

## 13 · Scatter Plot with Annotations

In [ ]:
plt.figure(figsize=(8, 5))

for _, row in df.iterrows():
    plt.scatter(row['cpu_hours'], row['cost'], s=120, zorder=5)
    plt.annotate(
        row['employee_id'],
        xy=(row['cpu_hours'], row['cost']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9
    )

# Trend line
z = np.polyfit(df['cpu_hours'], df['cost'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['cpu_hours'].min(), df['cpu_hours'].max(), 100)
plt.plot(x_line, p(x_line), 'r--', alpha=0.6, label='Trend')

plt.title('Cost vs CPU Hours')
plt.xlabel('CPU Hours')
plt.ylabel('Cost ($)')
plt.legend()
plt.tight_layout()
plt.show()

## 14 · Reading and Writing Files

In [ ]:
# Write to CSV
df.to_csv('chargeback_output.csv', index=False)
print("Saved: chargeback_output.csv")

# Read it back
df_reloaded = pd.read_csv('chargeback_output.csv')
print(f"Loaded {len(df_reloaded)} rows")
df_reloaded.head(3)

## 15 · Full Mini Analysis — Putting It All Together

In [ ]:
# ── Build dataset ───────────────────────────────────────────────
data = pd.DataFrame({
    'employee_id': ['C1234','F5678','C9012','F3456','C7890'],
    'department':  ['Quant','Risk','Quant','Tech','Risk'],
    'month':       ['2025-01','2025-01','2025-02','2025-02','2025-02'],
    'cost':        [1200.0, 850.0, 2100.0, 450.0, 3200.0],
    'cpu_hours':   [120, 80, 210, 45, 310],
    'gpu_hours':   [10, 0, 30, 5, 50],
})

data['total_hours'] = data['cpu_hours'] + data['gpu_hours']
data['cost_pct']    = (data['cost'] / data['cost'].sum() * 100).round(1)

# ── Summary ──────────────────────────────────────────────────────
summary = data.groupby('department').agg(
    users      = ('employee_id','nunique'),
    total_cost = ('cost','sum'),
    avg_cost   = ('cost','mean'),
).round(2)

# ── Dashboard chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('EMR Cluster Chargeback · Jan–Feb 2025',
             fontsize=13, fontweight='bold')

# Bar: cost by user
colors = ['#4472C4' if d=='Quant' else '#ED7D31' if d=='Risk'
          else '#A9D18E' for d in data['department']]
axes[0].barh(data['employee_id'], data['cost'], color=colors)
axes[0].set_title('Cost by User')
axes[0].set_xlabel('Cost ($)')

# Pie: share by department
dept = data.groupby('department')['cost'].sum()
axes[1].pie(dept, labels=dept.index, autopct='%1.0f%%', startangle=90)
axes[1].set_title('Share by Department')

# Scatter: cost vs hours
axes[2].scatter(data['cpu_hours'], data['cost'], s=100, color=colors)
for _, r in data.iterrows():
    axes[2].annotate(r['employee_id'],
                     (r['cpu_hours'], r['cost']),
                     textcoords='offset points', xytext=(4,4), fontsize=8)
axes[2].set_title('Cost vs CPU Hours')
axes[2].set_xlabel('CPU Hours')
axes[2].set_ylabel('Cost ($)')

plt.tight_layout()
plt.show()

# ── Final table ──────────────────────────────────────────────────
print("Summary:")
summary

---
## Quick Reference

| Action | Shortcut |
|---|---|
| Run cell + move to next | Shift + Enter |
| Run cell, stay | Ctrl + Enter |
| Insert cell above | Esc → A |
| Insert cell below | Esc → B |
| Delete cell | Esc → D D |
| Markdown cell | Esc → M |
| Code cell | Esc → Y |
| Restart kernel | Esc → 0 0 |
| Autocomplete | Tab |
| Show docs | Shift + Tab |
| Save | Ctrl/Cmd + S |

**Remember:** Always **Restart & Run All** before sharing a notebook!